# Final Model Improvement Experiments

This notebook investigates whether the predictive performance of the
selected forest-fire burned-area model can be improved further.

The current final model is a weighted ensemble consisting of:

- 70% HistGradientBoosting Regressor
- 30% Random Forest Regressor

The baseline model achieved:

- Test R²: 0.2302
- Test MAE: 0.8328
- Test RMSE: 1.0421

Although this model provides the strongest performance obtained so far,
previous error analysis showed substantial underprediction of rare
extreme-fire events.

Therefore, this notebook investigates alternative modelling strategies,
including advanced gradient boosting, alternative target transformations,
cross-validation, and severity-aware weighting.

The existing final model is retained as the baseline and will not be
modified.

The objective is to determine whether a statistically and scientifically
meaningful improvement can be achieved without introducing data leakage
or excessive model complexity.

In [1]:
# IMPORT LIBRARIES

import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

from sklearn.ensemble import (
    HistGradientBoostingRegressor,
    RandomForestRegressor
)

warnings.filterwarnings("ignore")

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
# LOAD PROCESSED DATA

PROJECT_ROOT = Path("../../")
PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed"

X_train = pd.read_csv(
    PROCESSED_DIR / "X_train.csv"
)

X_test = pd.read_csv(
    PROCESSED_DIR / "X_test.csv"
)

y_train = pd.read_csv(
    PROCESSED_DIR / "y_train.csv"
).squeeze("columns")

y_test = pd.read_csv(
    PROCESSED_DIR / "y_test.csv"
).squeeze("columns")

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (9042, 44)
X_test : (2261, 44)
y_train: (9042,)
y_test : (2261,)


In [3]:
# LOAD CURRENT BEST MODEL

MODEL_DIR = PROJECT_ROOT / "models"

baseline_hist = joblib.load(
    MODEL_DIR / "hist_gradient_boosting_final.pkl"
)

baseline_rf = joblib.load(
    MODEL_DIR / "random_forest_final.pkl"
)

baseline_hist_pred = baseline_hist.predict(X_test)
baseline_rf_pred = baseline_rf.predict(X_test)

baseline_pred = (
    0.70 * baseline_hist_pred
    + 0.30 * baseline_rf_pred
)

baseline_r2 = r2_score(
    y_test,
    baseline_pred
)

baseline_mae = mean_absolute_error(
    y_test,
    baseline_pred
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_pred
    )
)

print("=" * 55)
print("CURRENT BASELINE MODEL")
print("=" * 55)

print(f"R²   : {baseline_r2:.4f}")
print(f"MAE  : {baseline_mae:.4f}")
print(f"RMSE : {baseline_rmse:.4f}")

CURRENT BASELINE MODEL
R²   : 0.2302
MAE  : 0.8328
RMSE : 1.0421


## XGBoost

In [5]:
try:
    import xgboost as xgb

    print("XGBoost version:", xgb.__version__)
    print("XGBoost is available.")

except ImportError:
    print("XGBoost is NOT installed in the current environment.")

XGBoost version: 3.4.0
XGBoost is available.


In [6]:
# XGBOOST BASELINE MODEL

import xgboost as xgb

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.4.0


In [7]:
# TRAIN XGBOOST BASELINE

xgb_baseline = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

print("Training XGBoost...")

xgb_baseline.fit(
    X_train,
    y_train
)

print("XGBoost training completed.")

Training XGBoost...
XGBoost training completed.


In [8]:
# EVALUATE XGBOOST


xgb_train_pred = xgb_baseline.predict(X_train)
xgb_test_pred = xgb_baseline.predict(X_test)

xgb_train_r2 = r2_score(
    y_train,
    xgb_train_pred
)

xgb_test_r2 = r2_score(
    y_test,
    xgb_test_pred
)

xgb_test_mae = mean_absolute_error(
    y_test,
    xgb_test_pred
)

xgb_test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        xgb_test_pred
    )
)

print("=" * 55)
print("XGBOOST BASELINE RESULTS")
print("=" * 55)

print(f"Train R² : {xgb_train_r2:.4f}")
print(f"Test R²  : {xgb_test_r2:.4f}")
print(f"Test MAE : {xgb_test_mae:.4f}")
print(f"Test RMSE: {xgb_test_rmse:.4f}")

print()
print("CURRENT ENSEMBLE BASELINE")
print("=" * 55)

print(f"Test R²  : {baseline_r2:.4f}")
print(f"Test MAE : {baseline_mae:.4f}")
print(f"Test RMSE: {baseline_rmse:.4f}")

XGBOOST BASELINE RESULTS
Train R² : 0.6931
Test R²  : 0.2356
Test MAE : 0.8284
Test RMSE: 1.0385

CURRENT ENSEMBLE BASELINE
Test R²  : 0.2302
Test MAE : 0.8328
Test RMSE: 1.0421


### Interpretation

The baseline XGBoost model achieved a test R² of 0.2356, meaning it explains approximately 23.6% of the variation in the log-transformed burned-area target on unseen data. This represents a small improvement over the existing 70% HistGradientBoosting + 30% Random Forest ensemble, which achieved a test R² of 0.2302.

The XGBoost model also produced slightly lower errors, with a Test MAE of 0.8284 and Test RMSE of 1.0385, compared with 0.8328 and 1.0421 for the existing ensemble. Therefore, XGBoost currently provides the best individual model performance observed so far, although the improvement is relatively small.

The training R² of 0.6931 is considerably higher than the test R² of 0.2356, indicating a noticeable generalisation gap. This suggests that XGBoost is learning substantial patterns from the training data but is unable to reproduce all of those patterns reliably on unseen observations. This is consistent with the dataset's challenging characteristics, particularly its highly skewed burned-area distribution and relatively limited representation of extreme fire events.

### XGBoost Hyperparameter Tuning

In [9]:
import optuna

print("Optuna version:", optuna.__version__)
print("Optuna loaded successfully.")

Optuna version: 4.9.0
Optuna loaded successfully.


In [10]:
# XGBOOST ADVANCED TUNING SETUP

import optuna
import xgboost as xgb
import numpy as np

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

print("XGBoost version:", xgb.__version__)
print("Optuna version:", optuna.__version__)

print("Advanced tuning environment ready.")

XGBoost version: 3.4.0
Optuna version: 4.9.0
Advanced tuning environment ready.


In [11]:
# ADVANCED XGBOOST HYPERPARAMETER OPTIMIZATION

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

# 5-fold cross-validation
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

def objective(trial):

    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators",
            300,
            1500
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.005,
            0.08,
            log=True
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            2,
            10
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight",
            1,
            30
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.60,
            1.00
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.50,
            1.00
        ),

        "gamma": trial.suggest_float(
            "gamma",
            0.0,
            2.0
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            1e-8,
            10.0,
            log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            0.01,
            20.0,
            log=True
        ),

        "objective": "reg:squarederror",
        "random_state": 42,
        "n_jobs": -1
    }

    fold_scores = []

    for train_idx, val_idx in kf.split(X_train):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model = xgb.XGBRegressor(**params)

        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        val_pred = model.predict(X_val)

        fold_r2 = r2_score(
            y_val,
            val_pred
        )

        fold_scores.append(fold_r2)

    return np.mean(fold_scores)


# Create Optuna study
study = optuna.create_study(
    direction="maximize",
    study_name="forest_fire_xgboost_advanced"
)

print("=" * 60)
print("STARTING ADVANCED XGBOOST OPTIMIZATION")
print("=" * 60)
print("Cross-validation folds: 5")
print("Optimization trials: 150")
print()
print("This may take some time...")


study.optimize(
    objective,
    n_trials=150,
    show_progress_bar=True
)

print()
print("=" * 60)
print("OPTIMIZATION COMPLETED")
print("=" * 60)

[I 2026-08-10 14:58:41,857] A new study created in memory with name: forest_fire_xgboost_advanced


STARTING ADVANCED XGBOOST OPTIMIZATION
Cross-validation folds: 5
Optimization trials: 150

This may take some time...


  0%|          | 0/150 [00:00<?, ?it/s]

[I 2026-08-10 14:59:34,641] Trial 0 finished with value: 0.19096844660973758 and parameters: {'n_estimators': 1442, 'learning_rate': 0.032340801666427436, 'max_depth': 3, 'min_child_weight': 10, 'subsample': 0.6372122738001602, 'colsample_bytree': 0.6850606094209768, 'gamma': 0.5098648301766942, 'reg_alpha': 0.0009240914237750821, 'reg_lambda': 4.865372846352219}. Best is trial 0 with value: 0.19096844660973758.
[I 2026-08-10 15:00:24,907] Trial 1 finished with value: 0.18847488311573293 and parameters: {'n_estimators': 1391, 'learning_rate': 0.009692572977365983, 'max_depth': 3, 'min_child_weight': 25, 'subsample': 0.7082603751724512, 'colsample_bytree': 0.6697988567764375, 'gamma': 1.6553574301633909, 'reg_alpha': 4.4150906911786825, 'reg_lambda': 6.224772000780406}. Best is trial 0 with value: 0.19096844660973758.
[I 2026-08-10 15:02:43,927] Trial 2 finished with value: 0.20842565868966045 and parameters: {'n_estimators': 1265, 'learning_rate': 0.005308711257730296, 'max_depth': 9, 

In [12]:
# BEST XGBOOST CONFIGURATION

print("=" * 60)
print("BEST XGBOOST TUNING RESULT")
print("=" * 60)

print("Best 5-Fold CV R²:")
print(round(study.best_value, 5))

print("\nBest Parameters:")

for key, value in study.best_params.items():
    print(f"{key}: {value}")

BEST XGBOOST TUNING RESULT
Best 5-Fold CV R²:
0.21235

Best Parameters:
n_estimators: 1268
learning_rate: 0.008276102891538179
max_depth: 7
min_child_weight: 4
subsample: 0.867007492731578
colsample_bytree: 0.5892267514768841
gamma: 1.5964641892367197
reg_alpha: 3.0398616203941947e-05
reg_lambda: 12.90612737170908


### Interpretation

The optimization explored 150 configurations, and the best configuration achieved a cross-validation R² of 0.2124. This indicates that the tuned XGBoost model can learn meaningful relationships in the training data, but the improvement from hyperparameter optimization is limited.

The relatively low learning rate (0.0083) combined with a large number of trees (1,268) suggests that the optimization favoured gradual learning rather than aggressive boosting. The relatively strong reg_lambda value also indicates that additional regularization was useful for controlling model complexity.

Most importantly, the CV result does not indicate that we have found a dramatically better model. This supports our earlier observation that the main limitation is likely not simply the XGBoost hyperparameters, but the characteristics of the burned-area target and the difficulty of predicting rare extreme fires.

In [13]:
# TRAIN OPTIMIZED XGBOOST MODEL

best_xgb = xgb.XGBRegressor(
    **study.best_params,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

print("Training optimized XGBoost...")

best_xgb.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

print("Optimized XGBoost trained successfully.")

Training optimized XGBoost...
Optimized XGBoost trained successfully.


In [14]:
# FINAL TEST EVALUATION — OPTIMIZED XGBOOST

best_xgb_train_pred = best_xgb.predict(X_train)
best_xgb_test_pred = best_xgb.predict(X_test)

optimized_xgb_results = {
    "Model": "Optimized XGBoost",
    "Train R²": r2_score(
        y_train,
        best_xgb_train_pred
    ),
    "Test R²": r2_score(
        y_test,
        best_xgb_test_pred
    ),
    "Train MAE": mean_absolute_error(
        y_train,
        best_xgb_train_pred
    ),
    "Test MAE": mean_absolute_error(
        y_test,
        best_xgb_test_pred
    ),
    "Train RMSE": np.sqrt(
        mean_squared_error(
            y_train,
            best_xgb_train_pred
        )
    ),
    "Test RMSE": np.sqrt(
        mean_squared_error(
            y_test,
            best_xgb_test_pred
        )
    )
}

display(
    pd.DataFrame([optimized_xgb_results]).round(4)
)

,Model,Train R²,Test R²,Train MAE,Test MAE,Train RMSE,Test RMSE
0,Optimized XGBoost,0.6095,0.2451,0.5929,0.824,0.751,1.032


### Optimized XGBoost — Interpretation

The optimized XGBoost regression model achieved a Test R² of 0.2451, with a Test MAE of 0.8240 and a Test RMSE of 1.0320. Compared with the baseline XGBoost model, the optimization improved the Test R² from 0.2356 to 0.2451, while also reducing both MAE and RMSE. The optimized model therefore provides the best predictive performance obtained so far.

The model achieved a Training R² of 0.6095 compared with a Testing R² of 0.2451, indicating that the model captures substantially more variation in the training data than in unseen observations. This performance gap indicates some degree of overfitting, although the model still demonstrates meaningful generalisation to the test dataset.

Overall, hyperparameter optimization produced a moderate improvement rather than a dramatic increase in predictive performance. This suggests that the limitations of the model are not primarily caused by the choice of XGBoost hyperparameters. The highly skewed burned-area distribution and the limited number of extreme-fire observations remain important challenges, particularly for predicting very large burned areas.

Conclusion: The optimized XGBoost model was selected as the current best-performing individual regression model, achieving a Test R² of 0.2451, Test MAE of 0.8240, and Test RMSE of 1.0320. Further improvement should therefore focus on the underlying target distribution and the model's treatment of extreme-fire observations rather than additional conventional hyperparameter tuning.

### Target Transformation Experiment

In [15]:
# CHECK CURRENT TARGET TRANSFORMATION

print("Current target statistics")
print("=" * 50)

print("Minimum:", y_train.min())
print("Maximum:", y_train.max())
print("Mean   :", y_train.mean())
print("Median :", y_train.median())

print("\nFirst 10 target values:")
print(y_train.head(10))

Current target statistics
Minimum: 3.4339872044851463
Maximum: 11.586203807362043
Mean   : 5.132168644070621
Median : 4.890349128221754

First 10 target values:
0    4.543295
1    3.526361
2    6.511745
3    4.043051
4    7.540090
5    3.583519
6    4.927254
7    4.442651
8    6.061457
9    5.552960
Name: log_burned_area, dtype: float64


In [16]:
# CREATE ALTERNATIVE TARGET TRANSFORMATIONS

# Current target — already used by our best model
y_train_log = y_train.copy()
y_test_log = y_test.copy()

# Convert log target back to actual burned area
y_train_area = np.expm1(y_train)
y_test_area = np.expm1(y_test)

# Square-root transformation
y_train_sqrt = np.sqrt(y_train_area)
y_test_sqrt = np.sqrt(y_test_area)

# Raw burned area
y_train_raw = y_train_area.copy()
y_test_raw = y_test_area.copy()

print("Target transformations created successfully.")
print()
print("Log target range:")
print(round(y_train_log.min(), 4), "to", round(y_train_log.max(), 4))

print("\nSquare-root target range:")
print(round(y_train_sqrt.min(), 4), "to", round(y_train_sqrt.max(), 4))

print("\nRaw target range:")
print(round(y_train_raw.min(), 2), "to", round(y_train_raw.max(), 2))

Target transformations created successfully.

Log target range:
3.434 to 11.5862

Square-root target range:
5.4772 to 328.0274

Raw target range:
30.0 to 107602.0


In [17]:
# SQUARE-ROOT TARGET — OPTIMIZED XGBOOST

sqrt_xgb = xgb.XGBRegressor(
    **study.best_params,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

print("Training XGBoost with square-root target...")

sqrt_xgb.fit(
    X_train,
    y_train_sqrt,
    eval_set=[(X_test, y_test_sqrt)],
    verbose=False
)

print("Square-root XGBoost trained successfully.")

Training XGBoost with square-root target...
Square-root XGBoost trained successfully.


In [18]:
# EVALUATE SQUARE-ROOT XGBOOST

sqrt_train_pred = sqrt_xgb.predict(X_train)
sqrt_test_pred = sqrt_xgb.predict(X_test)

# Convert predictions back to hectares
sqrt_train_area_pred = np.square(sqrt_train_pred)
sqrt_test_area_pred = np.square(sqrt_test_pred)

# Prevent negative predictions
sqrt_train_area_pred = np.maximum(sqrt_train_area_pred, 0)
sqrt_test_area_pred = np.maximum(sqrt_test_area_pred, 0)

# Actual burned area
train_actual_area = np.expm1(y_train)
test_actual_area = np.expm1(y_test)

sqrt_results = {
    "Model": "Square-Root XGBoost",
    "Test R²": r2_score(
        test_actual_area,
        sqrt_test_area_pred
    ),
    "Test MAE": mean_absolute_error(
        test_actual_area,
        sqrt_test_area_pred
    ),
    "Test RMSE": np.sqrt(
        mean_squared_error(
            test_actual_area,
            sqrt_test_area_pred
        )
    )
}

display(
    pd.DataFrame([sqrt_results]).round(4)
)

,Model,Test R²,Test MAE,Test RMSE
0,Square-Root XGBoost,0.1016,373.4399,1269.4649


In [19]:
# FAIR RAW-SCALE COMPARISON
# LOG XGBOOST vs SQUARE-ROOT XGBOOST

# Optimized log-XGBoost predictions
log_test_pred = best_xgb.predict(X_test)

# Convert log predictions back to hectares
log_test_area_pred = np.expm1(log_test_pred)
log_test_area_pred = np.maximum(log_test_area_pred, 0)

# Actual burned area
actual_test_area = np.expm1(y_test)

# Log-model raw-scale metrics
log_raw_results = {
    "Model": "Optimized Log XGBoost",
    "Test R²": r2_score(
        actual_test_area,
        log_test_area_pred
    ),
    "Test MAE": mean_absolute_error(
        actual_test_area,
        log_test_area_pred
    ),
    "Test RMSE": np.sqrt(
        mean_squared_error(
            actual_test_area,
            log_test_area_pred
        )
    )
}

# Square-root model results already calculated
sqrt_raw_results = {
    "Model": "Square-Root XGBoost",
    "Test R²": r2_score(
        actual_test_area,
        sqrt_test_area_pred
    ),
    "Test MAE": mean_absolute_error(
        actual_test_area,
        sqrt_test_area_pred
    ),
    "Test RMSE": np.sqrt(
        mean_squared_error(
            actual_test_area,
            sqrt_test_area_pred
        )
    )
}

raw_scale_comparison = pd.DataFrame([
    log_raw_results,
    sqrt_raw_results
])

display(
    raw_scale_comparison.round(4)
)

,Model,Test R²,Test MAE,Test RMSE
0,Optimized Log XGBoost,0.0363,350.2122,1314.7881
1,Square-Root XGBoost,0.1016,373.4399,1269.4649


In [20]:
# TAIL-AWARE SAMPLE WEIGHTS

# Convert training target back to actual burned area
train_burned_area = np.expm1(y_train)

# Calculate thresholds using ONLY the training data
p90 = np.percentile(train_burned_area, 90)
p95 = np.percentile(train_burned_area, 95)
p99 = np.percentile(train_burned_area, 99)
p995 = np.percentile(train_burned_area, 99.5)

print("Tail-aware training thresholds:")
print(f"90th percentile : {p90:.4f} ha")
print(f"95th percentile : {p95:.4f} ha")
print(f"99th percentile : {p99:.4f} ha")
print(f"99.5th percentile: {p995:.4f} ha")

# Assign sample weights

sample_weights = np.ones(len(train_burned_area))

sample_weights[train_burned_area >= p90] = 1.5
sample_weights[train_burned_area >= p95] = 2.0
sample_weights[train_burned_area >= p99] = 3.0
sample_weights[train_burned_area >= p995] = 4.0

# Inspect weight distribution

unique_weights, weight_counts = np.unique(
    sample_weights,
    return_counts=True
)

weight_summary = pd.DataFrame({
    "Weight": unique_weights,
    "Records": weight_counts
})

print("\nSample-weight distribution:")
display(weight_summary)

Tail-aware training thresholds:
90th percentile : 871.0000 ha
95th percentile : 1665.0000 ha
99th percentile : 5930.9100 ha
99.5th percentile: 10373.1550 ha

Sample-weight distribution:


,Weight,Records
0,1.0,8136
1,1.5,452
2,2.0,363
3,3.0,45
4,4.0,46


### Train the Tail-Aware XGBoos

In [21]:
# TAIL-AWARE OPTIMIZED XGBOOST

tail_xgb = xgb.XGBRegressor(
    **study.best_params,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

print("Training tail-aware XGBoost...")

tail_xgb.fit(
    X_train,
    y_train,
    sample_weight=sample_weights,
    eval_set=[(X_test, y_test)],
    verbose=False
)

print("Tail-aware XGBoost trained successfully.")

Training tail-aware XGBoost...
Tail-aware XGBoost trained successfully.


In [22]:
# EVALUATE TAIL-AWARE XGBOOST

tail_train_pred = tail_xgb.predict(X_train)
tail_test_pred = tail_xgb.predict(X_test)

tail_results = {
    "Model": "Tail-Aware XGBoost",
    
    "Train R²": r2_score(
        y_train,
        tail_train_pred
    ),
    
    "Test R²": r2_score(
        y_test,
        tail_test_pred
    ),
    
    "Train MAE": mean_absolute_error(
        y_train,
        tail_train_pred
    ),
    
    "Test MAE": mean_absolute_error(
        y_test,
        tail_test_pred
    ),
    
    "Train RMSE": np.sqrt(
        mean_squared_error(
            y_train,
            tail_train_pred
        )
    ),
    
    "Test RMSE": np.sqrt(
        mean_squared_error(
            y_test,
            tail_test_pred
        )
    )
}

display(
    pd.DataFrame([tail_results]).round(4)
)

,Model,Train R²,Test R²,Train MAE,Test MAE,Train RMSE,Test RMSE
0,Tail-Aware XGBoost,0.6702,0.2355,0.5654,0.8345,0.6902,1.0385


### Tail-Aware XGBoost — Interpretation

The tail-aware XGBoost model was developed by assigning higher training weights to rare large-fire observations in order to improve the prediction of extreme burned areas. The weighting strategy assigned progressively larger weights to observations above the 90th, 95th, 99th, and 99.5th percentiles of burned area.

The tail-aware model achieved a Training R² of 0.6702 and a Testing R² of 0.2355. Compared with the optimized XGBoost model, which achieved a Testing R² of 0.2451, the tail-aware approach resulted in a slight reduction in overall predictive performance. The Test MAE increased from 0.8240 to 0.8345, while the Test RMSE increased from 1.0320 to 1.0385.

The higher Training R² indicates that the weighted model was able to fit the training observations more strongly, particularly the observations representing larger fires. However, this improvement did not generalise to the unseen test data. This suggests that increasing the importance of rare extreme-fire observations alone is insufficient to overcome the difficulty of predicting extreme burned areas.

Therefore, the tail-aware weighting strategy was not selected as the final model. The Optimized XGBoost model remains the best-performing model, with a Testing R² of 0.2451, Test MAE of 0.8240, and Test RMSE of 1.0320.

This experiment nevertheless provides an important finding: the limitations of the prediction system are not solved simply by assigning greater importance to extreme-fire observations.

### Analyse the Fire-Size Ranges

In [23]:
# FIRE-SIZE RANGE COMPARISON
# OPTIMIZED XGBOOST vs TAIL-AWARE XGBOOST

# Convert predictions back to hectares
optimized_pred_ha = np.expm1(best_xgb.predict(X_test))
tail_pred_ha = np.expm1(tail_xgb.predict(X_test))

# Actual burned area
actual_ha = np.expm1(y_test)

# Prevent negative predictions
optimized_pred_ha = np.maximum(optimized_pred_ha, 0)
tail_pred_ha = np.maximum(tail_pred_ha, 0)

# Create fire-size ranges
def get_fire_range(area):
    if area <= 100:
        return "30–100 ha"
    elif area <= 500:
        return "101–500 ha"
    elif area <= 5000:
        return "501–5,000 ha"
    elif area <= 10000:
        return "5,001–10,000 ha"
    else:
        return ">10,000 ha"


range_labels = actual_ha.apply(get_fire_range)

range_order = [
    "30–100 ha",
    "101–500 ha",
    "501–5,000 ha",
    "5,001–10,000 ha",
    ">10,000 ha"
]

range_results = []

for fire_range in range_order:

    mask = range_labels == fire_range

    actual_range = actual_ha[mask]

    optimized_range = optimized_pred_ha[mask]
    tail_range = tail_pred_ha[mask]

    range_results.append({
        "Fire Range": fire_range,
        "Records": mask.sum(),

        "Actual Mean (ha)": actual_range.mean(),

        "Optimized Mean Predicted (ha)": optimized_range.mean(),

        "Tail-Aware Mean Predicted (ha)": tail_range.mean(),

        "Optimized MAE (ha)": mean_absolute_error(
            actual_range,
            optimized_range
        ),

        "Tail-Aware MAE (ha)": mean_absolute_error(
            actual_range,
            tail_range
        )
    })

range_comparison = pd.DataFrame(range_results)

display(
    range_comparison.round(2)
)

,Fire Range,Records,Actual Mean (ha),Optimized Mean Predicted (ha),Tail-Aware Mean Predicted (ha),Optimized MAE (ha),Tail-Aware MAE (ha)
0,30–100 ha,918,58.14,144.350006,157.679993,86.62,99.99
1,101–500 ha,971,222.15,200.210007,228.860001,108.19,123.94
2,"501–5,000 ha",347,1301.07,293.619995,357.799988,1013.32,965.99
3,"5,001–10,000 ha",15,7128.47,494.170013,655.559998,6634.30,6472.91
4,">10,000 ha",10,16222.80,610.489990,1028.250000,15612.31,15194.55


### Fire-Size Range Analysis — Interpretation

The fire-size range analysis was conducted to determine how the optimized XGBoost and tail-aware XGBoost models perform across different levels of burned-area severity.

For smaller fires between 30–100 ha, the optimized XGBoost model predicted an average burned area of approximately 144.35 ha compared with an actual average of 58.14 ha. Although the model overpredicted this range, its MAE remained relatively moderate at 86.62 ha.

For fires between 101–500 ha, the optimized model produced an average prediction of 200.21 ha compared with an actual average of 222.15 ha, showing considerably better agreement.

However, prediction accuracy deteriorated substantially as fire severity increased. For the 501–5,000 ha range, the actual mean burned area was 1,301.07 ha, while the optimized model predicted only 293.62 ha on average. This demonstrates significant underprediction of larger fires.

The problem becomes particularly severe for extreme fires. For the 5,001–10,000 ha range, the actual mean was 7,128.47 ha, while the optimized model predicted only 494.17 ha. For fires exceeding 10,000 ha, the actual mean was 16,222.80 ha, whereas the optimized model predicted only 610.49 ha.

The tail-aware model increased predictions for the larger-fire categories, with mean predictions of 357.80 ha, 655.56 ha, and 1,028.25 ha respectively. However, these predictions remained substantially below the actual burned areas, and the tail-aware model did not improve overall test performance.

These results demonstrate that the primary weakness of the current prediction system is not the prediction of ordinary fires, but the inability to accurately estimate extremely large burned areas. The available environmental and meteorological features appear insufficient for the current regression models to reliably distinguish the magnitude of these rare extreme events.

Therefore, the optimized XGBoost model remains the best overall model, while the severe underprediction of extreme fires is identified as the major limitation of the prediction system.

### Robust XGBoost

In [24]:
# ROBUST XGBOOST — PSEUDO-HUBER LOSS

robust_xgb = xgb.XGBRegressor(
    **study.best_params,
    objective="reg:pseudohubererror",
    random_state=42,
    n_jobs=-1
)

print("Training Robust XGBoost with Pseudo-Huber loss...")

robust_xgb.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

print("Robust XGBoost trained successfully.")

Training Robust XGBoost with Pseudo-Huber loss...
Robust XGBoost trained successfully.


In [25]:
# EVALUATE ROBUST XGBOOST

robust_train_pred = robust_xgb.predict(X_train)
robust_test_pred = robust_xgb.predict(X_test)

robust_results = {
    "Model": "Robust XGBoost",
    
    "Train R²": r2_score(
        y_train,
        robust_train_pred
    ),
    
    "Test R²": r2_score(
        y_test,
        robust_test_pred
    ),
    
    "Train MAE": mean_absolute_error(
        y_train,
        robust_train_pred
    ),
    
    "Test MAE": mean_absolute_error(
        y_test,
        robust_test_pred
    ),
    
    "Train RMSE": np.sqrt(
        mean_squared_error(
            y_train,
            robust_train_pred
        )
    ),
    
    "Test RMSE": np.sqrt(
        mean_squared_error(
            y_test,
            robust_test_pred
        )
    )
}

display(
    pd.DataFrame([robust_results]).round(4)
)

,Model,Train R²,Test R²,Train MAE,Test MAE,Train RMSE,Test RMSE
0,Robust XGBoost,0.4537,0.2323,0.6744,0.8221,0.8883,1.0407


The Robust XGBoost model was not selected because its Test R² and RMSE were worse than the optimized XGBoost model. Although it achieved a marginally lower MAE, the overall predictive performance remained inferior. The Optimized XGBoost therefore remains the best model with a Test R² of 0.2451.

### Raw Burned-Area XGBoost

In [26]:
# SECTION 6.1 — RAW BURNED-AREA TARGET

# Convert the current log target back to hectares
y_train_raw = np.expm1(y_train)
y_test_raw = np.expm1(y_test)

print("=" * 60)
print("RAW BURNED-AREA TARGET")
print("=" * 60)

print(f"Training target range : {y_train_raw.min():.2f} to {y_train_raw.max():.2f} ha")
print(f"Testing target range  : {y_test_raw.min():.2f} to {y_test_raw.max():.2f} ha")

print(f"\nTraining mean   : {y_train_raw.mean():.2f} ha")
print(f"Training median : {y_train_raw.median():.2f} ha")

print(f"\nTesting mean    : {y_test_raw.mean():.2f} ha")
print(f"Testing median  : {y_test_raw.median():.2f} ha")

RAW BURNED-AREA TARGET
Training target range : 30.00 to 107602.00 ha
Testing target range  : 30.00 to 23078.00 ha

Training mean   : 502.26 ha
Training median : 132.00 ha

Testing mean    : 437.73 ha
Testing median  : 126.00 ha


### Raw Burned-Area Target — Interpretation

The original burned-area target was restored from the logarithmic transformation to evaluate whether XGBoost can learn the burned-area magnitude directly in hectares.

The training target ranges from 30 ha to 10,760 ha, while the testing target ranges from 30 ha to 23,078 ha. The training mean is 502.26 ha compared with a median of only 132.00 ha, demonstrating a strong right-skewed distribution caused by a relatively small number of very large fire events.

A similar pattern is observed in the testing data, where the mean burned area is 437.73 ha while the median is 126.00 ha. This substantial difference between the mean and median confirms that extreme fire events have a strong influence on the overall distribution.

The raw target therefore preserves the full magnitude of extreme fires but presents a considerably more difficult regression problem than the logarithmic target. This experiment will determine whether XGBoost can use the additional magnitude information to improve prediction accuracy in hectares.

### Train Raw-Target XGBoost

In [27]:
# SECTION 6.2 — RAW-TARGET XGBOOST

raw_xgb = xgb.XGBRegressor(
    **study.best_params,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

print("=" * 60)
print("TRAINING RAW-TARGET XGBOOST")
print("=" * 60)

raw_xgb.fit(
    X_train,
    y_train_raw,
    eval_set=[(X_test, y_test_raw)],
    verbose=False
)

print("Raw-target XGBoost trained successfully.")

TRAINING RAW-TARGET XGBOOST
Raw-target XGBoost trained successfully.


### Interpretation: Raw-Target XGBoost

The XGBoost regression model was successfully trained using the original burned-area values in hectares as the target variable. Unlike the previous logarithmic model, this approach does not compress the large-fire observations, allowing the model to directly learn the actual magnitude of burned areas.

The model uses the same optimized XGBoost hyperparameters identified during the previous tuning stage, ensuring that the effect of changing the target representation can be evaluated independently of additional hyperparameter changes.

The next evaluation will determine whether learning the raw burned-area values improves prediction accuracy for actual burned area, particularly for the large and extreme fire events that were substantially underpredicted by the logarithmic model.

In [28]:
# SECTION 6.3 — EVALUATE RAW-TARGET XGBOOST

raw_train_pred = raw_xgb.predict(X_train)
raw_test_pred = raw_xgb.predict(X_test)

# Prevent negative burned-area predictions
raw_train_pred = np.maximum(raw_train_pred, 0)
raw_test_pred = np.maximum(raw_test_pred, 0)

raw_results = {
    "Model": "Raw-Target XGBoost",
    "Train R²": r2_score(
        y_train_raw,
        raw_train_pred
    ),
    "Test R²": r2_score(
        y_test_raw,
        raw_test_pred
    ),
    "Train MAE": mean_absolute_error(
        y_train_raw,
        raw_train_pred
    ),
    "Test MAE": mean_absolute_error(
        y_test_raw,
        raw_test_pred
    ),
    "Train RMSE": np.sqrt(
        mean_squared_error(
            y_train_raw,
            raw_train_pred
        )
    ),
    "Test RMSE": np.sqrt(
        mean_squared_error(
            y_test_raw,
            raw_test_pred
        )
    )
}

display(
    pd.DataFrame([raw_results]).round(4)
)

,Model,Train R²,Test R²,Train MAE,Test MAE,Train RMSE,Test RMSE
0,Raw-Target XGBoost,0.7592,0.0567,323.4023,486.8215,1100.2723,1300.7959


### Interpretation: Raw-Target XGBoost

The raw-target XGBoost model was evaluated directly on burned area measured in hectares. The model achieved a Training R² of 0.7592 but only a Testing R² of 0.0567. The Test MAE was 486.82 ha and the Test RMSE was 1,300.80 ha.

Although the relatively high Training R² indicates that the model can fit the training observations effectively, the much lower Testing R² demonstrates a substantial generalisation gap. This indicates that learning the raw burned-area values directly causes the model to fit the training data without reliably generalising to unseen observations.

The large Test RMSE also demonstrates the strong influence of extreme fire events on the raw-scale prediction error. Since the burned-area distribution is highly right-skewed, the relatively small number of very large fires creates a difficult regression problem when the target is represented directly in hectares.

The raw-target approach therefore did not improve prediction accuracy and performed worse than both the optimized logarithmic XGBoost and square-root XGBoost approaches. The logarithmic transformation remains preferable for the main regression model because it provides a more stable learning problem.

The raw-target experiment confirms that simply removing the logarithmic transformation does not solve the extreme-fire prediction problem.

### Environmental Feature Interaction Engineering

#### Create Environmental Interaction Features

In [29]:
# SECTION 7.1 — ENVIRONMENTAL INTERACTION FEATURES

print("=" * 60)
print("CREATING ENVIRONMENTAL INTERACTION FEATURES")
print("=" * 60)

# Make copies so the original datasets remain unchanged
X_train_interact = X_train.copy()
X_test_interact = X_test.copy()

# Temperature × Wind

X_train_interact["temp_wind_interaction"] = (
    X_train_interact["temperature_c"] *
    X_train_interact["wind_speed"]
)

X_test_interact["temp_wind_interaction"] = (
    X_test_interact["temperature_c"] *
    X_test_interact["wind_speed"]
)

# Temperature × Relative Humidity

X_train_interact["temp_humidity_interaction"] = (
    X_train_interact["temperature_c"] *
    X_train_interact["relative_humidity"]
)

X_test_interact["temp_humidity_interaction"] = (
    X_test_interact["temperature_c"] *
    X_test_interact["relative_humidity"]
)

# Temperature × Soil Moisture

X_train_interact["temp_soil_moisture_interaction"] = (
    X_train_interact["temperature_c"] *
    X_train_interact["soil_moisture"]
)

X_test_interact["temp_soil_moisture_interaction"] = (
    X_test_interact["temperature_c"] *
    X_test_interact["soil_moisture"]
)

# Wind × Soil Moisture

X_train_interact["wind_soil_moisture_interaction"] = (
    X_train_interact["wind_speed"] *
    X_train_interact["soil_moisture"]
)

X_test_interact["wind_soil_moisture_interaction"] = (
    X_test_interact["wind_speed"] *
    X_test_interact["soil_moisture"]
)

# Temperature × NDVI

X_train_interact["temp_ndvi_interaction"] = (
    X_train_interact["temperature_c"] *
    X_train_interact["ndvi"]
)

X_test_interact["temp_ndvi_interaction"] = (
    X_test_interact["temperature_c"] *
    X_test_interact["ndvi"]
)

# Wind × Relative Humidity

X_train_interact["wind_humidity_interaction"] = (
    X_train_interact["wind_speed"] *
    X_train_interact["relative_humidity"]
)

X_test_interact["wind_humidity_interaction"] = (
    X_test_interact["wind_speed"] *
    X_test_interact["relative_humidity"]
)


print("Environmental interaction features created successfully.")

print("\nOriginal feature count:", X_train.shape[1])
print("New feature count:", X_train_interact.shape[1])

print("\nNew interaction features:")
print([
    "temp_wind_interaction",
    "temp_humidity_interaction",
    "temp_soil_moisture_interaction",
    "wind_soil_moisture_interaction",
    "temp_ndvi_interaction",
    "wind_humidity_interaction"
])

CREATING ENVIRONMENTAL INTERACTION FEATURES
Environmental interaction features created successfully.

Original feature count: 44
New feature count: 50

New interaction features:
['temp_wind_interaction', 'temp_humidity_interaction', 'temp_soil_moisture_interaction', 'wind_soil_moisture_interaction', 'temp_ndvi_interaction', 'wind_humidity_interaction']


### Interpretation: Environmental Interaction Features

Six additional environmental interaction features were successfully created, increasing the feature count from 44 to 50.

The new features represent relationships between important environmental variables, including temperature, wind speed, relative humidity, soil moisture, and vegetation conditions. These interactions are intended to provide the model with additional information about combined environmental conditions that may contribute to increased fire severity.

The interaction features include temperature–wind, temperature–humidity, temperature–soil-moisture, wind–soil-moisture, temperature–NDVI, and wind–humidity relationships.

The original feature set was preserved, allowing the interaction-enhanced model to be evaluated fairly against the existing Optimized XGBoost benchmark. The new features will only be retained if they produce an improvement on unseen test data.

In [30]:
# SECTION 7.2 — VALIDATE INTERACTION FEATURES

interaction_features = [
    "temp_wind_interaction",
    "temp_humidity_interaction",
    "temp_soil_moisture_interaction",
    "wind_soil_moisture_interaction",
    "temp_ndvi_interaction",
    "wind_humidity_interaction"
]

interaction_check = pd.DataFrame({
    "Feature": interaction_features,
    "Train_Missing": [
        X_train_interact[f].isna().sum()
        for f in interaction_features
    ],
    "Test_Missing": [
        X_test_interact[f].isna().sum()
        for f in interaction_features
    ],
    "Train_Infinite": [
        np.isinf(X_train_interact[f]).sum()
        for f in interaction_features
    ],
    "Test_Infinite": [
        np.isinf(X_test_interact[f]).sum()
        for f in interaction_features
    ]
})

display(interaction_check)

print("\nInteraction feature statistics:")
display(
    X_train_interact[interaction_features]
    .describe()
    .T
    .round(4)
)

,Feature,Train_Missing,Test_Missing,Train_Infinite,Test_Infinite
0,temp_wind_interaction,0,0,0,0
1,temp_humidity_interaction,0,0,0,0
2,temp_soil_moisture_interaction,0,0,0,0
3,wind_soil_moisture_interaction,0,0,0,0
4,temp_ndvi_interaction,0,0,0,0
5,wind_humidity_interaction,0,0,0,0



Interaction feature statistics:


,count,mean,std,min,25%,50%,75%,max
temp_wind_interaction,9042.0,92.8976,45.8524,-2.5361,61.4502,92.1737,116.6207,382.7388
temp_humidity_interaction,9042.0,793.3866,264.6353,-35.5879,610.4975,810.1660,919.6572,1871.9006
temp_soil_moisture_interaction,9042.0,8.8741,3.8163,-0.7670,6.9319,8.5054,10.2562,33.5876
wind_soil_moisture_interaction,9042.0,1.1775,0.8158,-0.0238,0.7068,0.9318,1.3953,9.0074
temp_ndvi_interaction,9042.0,12.9835,5.5077,-0.3686,8.9191,12.6491,16.8771,32.3917
wind_humidity_interaction,9042.0,104.2428,62.5782,10.5963,64.7206,95.5999,122.0730,793.9255


### Interpretation: Interaction Feature Validation

All six newly created interaction features were successfully validated before model training. No missing values or infinite values were detected in either the training or testing datasets.

The interaction features also show reasonable numerical distributions without obvious computational anomalies. For example, the temperature–wind interaction ranges from approximately -2.54 to 382.74, while the wind–humidity interaction ranges from approximately 10.60 to 793.93.

The successful validation confirms that the engineered features can be safely introduced into the XGBoost model without requiring additional missing-value or infinity handling.

The interaction-enhanced feature set therefore contains 50 valid predictors and is ready for model training. Its performance will be compared directly against the existing 44-feature Optimized XGBoost model to determine whether the additional environmental relationships improve generalisation.

### Train Interaction-Enhanced XGBoost

In [ ]:
# INTERACTION-ENHANCED XGBOOST

interaction_xgb = xgb.XGBRegressor(
    **study.best_params,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

print("=" * 60)
print("TRAINING INTERACTION-ENHANCED XGBOOST")
print("=" * 60)

interaction_xgb.fit(
    X_train_interact,
    y_train,
    eval_set=[(X_test_interact, y_test)],
    verbose=False
)

print("Interaction-enhanced XGBoost trained successfully.")

TRAINING INTERACTION-ENHANCED XGBOOST
Interaction-enhanced XGBoost trained successfully.


In [32]:
# EVALUATE INTERACTION-ENHANCED XGBOOST

interaction_train_pred = interaction_xgb.predict(X_train_interact)
interaction_test_pred = interaction_xgb.predict(X_test_interact)

interaction_results = {
    "Model": "Interaction-Enhanced XGBoost",

    "Train R²": r2_score(
        y_train,
        interaction_train_pred
    ),

    "Test R²": r2_score(
        y_test,
        interaction_test_pred
    ),

    "Train MAE": mean_absolute_error(
        y_train,
        interaction_train_pred
    ),

    "Test MAE": mean_absolute_error(
        y_test,
        interaction_test_pred
    ),

    "Train RMSE": np.sqrt(
        mean_squared_error(
            y_train,
            interaction_train_pred
        )
    ),

    "Test RMSE": np.sqrt(
        mean_squared_error(
            y_test,
            interaction_test_pred
        )
    )
}

display(
    pd.DataFrame([interaction_results]).round(4)
)

,Model,Train R²,Test R²,Train MAE,Test MAE,Train RMSE,Test RMSE
0,Interaction-Enhanced XGBoost,0.6225,0.237,0.5828,0.8285,0.7384,1.0375


### Interpretation: Interaction-Enhanced XGBoost

The interaction-enhanced XGBoost model was evaluated using the same test dataset and the same optimized hyperparameters as the baseline model. The addition of six environmental interaction features increased the feature count from 44 to 50.

The model achieved a Training R² of 0.6225 and a Testing R² of 0.2370. The Test MAE was 0.8285 and the Test RMSE was 1.0375.

Compared with the original Optimized XGBoost model, which achieved a Testing R² of 0.2451, Test MAE of 0.8240, and Test RMSE of 1.0320, the interaction-enhanced model produced slightly worse performance across all three test metrics.

The result indicates that the manually engineered temperature, humidity, wind, soil-moisture, and NDVI interaction features did not provide additional predictive information that generalized effectively to the unseen test data. Although the training R² increased slightly, the decrease in testing performance suggests that the additional features may introduce redundant information or mild overfitting.

Therefore, the six manually engineered interaction features will not be included in the final model. The original Optimized XGBoost remains the current best-performing model.

### Select Top Features

In [ ]:
# TOP FEATURE SELECTION

top_features = [
    "relative_humidity",
    "year",
    "longitude",
    "latitude",
    "day_of_year_cos",
    "wind_speed",
    "lc_agriculture",
    "ndvi",
    "temperature_c",
    "population",
    "lai",
    "soil_moisture",
    "roads_distance_km",
    "elevation",
    "lc_forest",
    "slope_degrees",
    "surface_pressure",
    "lc_shrubland",
    "day_of_year_sin",
    "day_of_year"
]

print("=" * 60)
print("TOP 20 FEATURES SELECTED")
print("=" * 60)

for i, feature in enumerate(top_features, start=1):
    print(f"{i:2d}. {feature}")

print("\nOriginal feature count:", X_train.shape[1])
print("Selected feature count:", len(top_features))

TOP 20 FEATURES SELECTED
 1. relative_humidity
 2. year
 3. longitude
 4. latitude
 5. day_of_year_cos
 6. wind_speed
 7. lc_agriculture
 8. ndvi
 9. temperature_c
10. population
11. lai
12. soil_moisture
13. roads_distance_km
14. elevation
15. lc_forest
16. slope_degrees
17. surface_pressure
18. lc_shrubland
19. day_of_year_sin
20. day_of_year

Original feature count: 44
Selected feature count: 20


### Interpretation: Top Feature Selection

The permutation-importance analysis identified 20 features with the strongest contribution to the model's predictive performance. These include relative humidity, year, longitude, latitude, seasonal timing, wind speed, land-cover information, NDVI, temperature, population, LAI, soil moisture, elevation, road distance, and surface pressure.

The feature set was reduced from 44 original predictors to 20 selected predictors. This reduction removes lower-contribution variables while retaining the features that demonstrated the greatest importance in the previous permutation-importance analysis.

The purpose of this experiment is to determine whether a simpler feature set can improve model generalisation by reducing redundant or weak predictors. The original 44-feature model remains unchanged and will continue to serve as the benchmark.

In [ ]:
# PREPARE TOP-20 FEATURE DATASET

X_train_top20 = X_train[top_features].copy()
X_test_top20 = X_test[top_features].copy()

print("=" * 60)
print("TOP-20 FEATURE DATASET")
print("=" * 60)

print("Training shape:", X_train_top20.shape)
print("Testing shape :", X_test_top20.shape)

print("\nMissing values:")
print("Training:", X_train_top20.isna().sum().sum())
print("Testing :", X_test_top20.isna().sum().sum())

print("\nSelected features:")
print(X_train_top20.columns.tolist())

TOP-20 FEATURE DATASET
Training shape: (9042, 20)
Testing shape : (2261, 20)

Missing values:
Training: 0
Testing : 0

Selected features:
['relative_humidity', 'year', 'longitude', 'latitude', 'day_of_year_cos', 'wind_speed', 'lc_agriculture', 'ndvi', 'temperature_c', 'population', 'lai', 'soil_moisture', 'roads_distance_km', 'elevation', 'lc_forest', 'slope_degrees', 'surface_pressure', 'lc_shrubland', 'day_of_year_sin', 'day_of_year']


### Interpretation: Top-20 Dataset Preparation

The selected feature set was successfully prepared using the 20 most important predictors identified through permutation importance. The resulting training dataset contains 9,042 observations and 20 features, while the testing dataset contains 2,261 observations and the same 20 features.

No missing values were detected in either dataset, confirming that the reduced feature set is suitable for model training without requiring additional preprocessing.

The reduction from 44 to 20 predictors provides a controlled feature-selection experiment. The original 44-feature model remains unchanged and will be used as the benchmark to determine whether removing lower-importance features improves the model's ability to generalise to unseen data.

In [ ]:
# TOP-20 OPTIMIZED XGBOOST

top20_xgb = xgb.XGBRegressor(
    **study.best_params,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

print("=" * 60)
print("TRAINING TOP-20 OPTIMIZED XGBOOST")
print("=" * 60)

top20_xgb.fit(
    X_train_top20,
    y_train,
    eval_set=[(X_test_top20, y_test)],
    verbose=False
)

print("Top-20 Optimized XGBoost trained successfully.")

TRAINING TOP-20 OPTIMIZED XGBOOST
Top-20 Optimized XGBoost trained successfully.


In [ ]:
# EVALUATE TOP-20 OPTIMIZED XGBOOST

top20_train_pred = top20_xgb.predict(X_train_top20)
top20_test_pred = top20_xgb.predict(X_test_top20)

top20_results = {
    "Model": "Top-20 Optimized XGBoost",

    "Train R²": r2_score(
        y_train,
        top20_train_pred
    ),

    "Test R²": r2_score(
        y_test,
        top20_test_pred
    ),

    "Train MAE": mean_absolute_error(
        y_train,
        top20_train_pred
    ),

    "Test MAE": mean_absolute_error(
        y_test,
        top20_test_pred
    ),

    "Train RMSE": np.sqrt(
        mean_squared_error(
            y_train,
            top20_train_pred
        )
    ),

    "Test RMSE": np.sqrt(
        mean_squared_error(
            y_test,
            top20_test_pred
        )
    )
}

display(
    pd.DataFrame([top20_results]).round(4)
)

,Model,Train R²,Test R²,Train MAE,Test MAE,Train RMSE,Test RMSE
0,Top-20 Optimized XGBoost,0.5706,0.2428,0.6234,0.8249,0.7876,1.0336


### Interpretation: Top-20 Optimized XGBoost

The Top-20 Optimized XGBoost model was evaluated using the 20 most important features identified through permutation importance. The model achieved a Training R² of 0.5706 and a Testing R² of 0.2428. The Test MAE was 0.8249 and the Test RMSE was 1.0336.

Compared with the original 44-feature Optimized XGBoost model, which achieved a Testing R² of 0.2451, Test MAE of 0.8240, and Test RMSE of 1.0320, the reduced feature model produced slightly lower performance across all three test metrics.

The reduction from 44 to 20 features therefore did not provide an improvement in predictive accuracy. Although the selected features contained most of the model's important predictive information, the remaining features appear to provide a small amount of additional information that contributes to generalisation.

The Top-20 model was therefore not selected as the final model. The original 44-feature Optimized XGBoost remains the current best-performing model.

In [ ]:
# TRAINING VS TESTING TARGET DISTRIBUTION

train_distribution = {
    "Dataset": "Training",
    "Minimum (ha)": y_train_raw.min(),
    "25th Percentile (ha)": np.percentile(y_train_raw, 25),
    "Median (ha)": np.percentile(y_train_raw, 50),
    "Mean (ha)": y_train_raw.mean(),
    "75th Percentile (ha)": np.percentile(y_train_raw, 75),
    "90th Percentile (ha)": np.percentile(y_train_raw, 90),
    "95th Percentile (ha)": np.percentile(y_train_raw, 95),
    "99th Percentile (ha)": np.percentile(y_train_raw, 99),
    "Maximum (ha)": y_train_raw.max()
}

test_distribution = {
    "Dataset": "Testing",
    "Minimum (ha)": y_test_raw.min(),
    "25th Percentile (ha)": np.percentile(y_test_raw, 25),
    "Median (ha)": np.percentile(y_test_raw, 50),
    "Mean (ha)": y_test_raw.mean(),
    "75th Percentile (ha)": np.percentile(y_test_raw, 75),
    "90th Percentile (ha)": np.percentile(y_test_raw, 90),
    "95th Percentile (ha)": np.percentile(y_test_raw, 95),
    "99th Percentile (ha)": np.percentile(y_test_raw, 99),
    "Maximum (ha)": y_test_raw.max()
}

distribution_comparison = pd.DataFrame([
    train_distribution,
    test_distribution
])

display(
    distribution_comparison.round(2)
)

,Dataset,Minimum (ha),25th Percentile (ha),Median (ha),Mean (ha),75th Percentile (ha),90th Percentile (ha),95th Percentile (ha),99th Percentile (ha),Maximum (ha)
0,Training,30.0,66.0,132.0,502.26,340.75,871.0,1665.0,5930.91,107602.0
1,Testing,30.0,63.0,126.0,437.73,319.00,852.0,1647.0,5826.40,23078.0


In [ ]:
# EXTREME FIRE COVERAGE

thresholds = [500, 1000, 5000, 10000, 15000, 20000]

extreme_comparison = []

for threshold in thresholds:

    train_count = np.sum(y_train_raw > threshold)
    test_count = np.sum(y_test_raw > threshold)

    extreme_comparison.append({
        "Threshold (ha)": threshold,
        "Training Count": train_count,
        "Testing Count": test_count,
        "Training %": 100 * train_count / len(y_train_raw),
        "Testing %": 100 * test_count / len(y_test_raw)
    })

extreme_comparison = pd.DataFrame(extreme_comparison)

display(
    extreme_comparison.round(3)
)

,Threshold (ha),Training Count,Testing Count,Training %,Testing %
0,500,1616,372,17.872,16.453
1,1000,782,193,8.649,8.536
2,5000,119,25,1.316,1.106
3,10000,48,10,0.531,0.442
4,15000,26,6,0.288,0.265
5,20000,16,3,0.177,0.133


### Interpretation: Training vs Testing Target Distribution

The comparison between the training and testing target distributions shows that the two datasets have broadly similar burned-area distributions.

The training median burned area is 132 ha, compared with 126 ha for the testing set, while the training mean is 502.26 ha compared with 437.73 ha for testing. The 90th, 95th, and 99th percentiles are also relatively similar between the two datasets.

Although the maximum testing value is 23,078 ha compared with 10,760 ha in the training set, the extreme-fire analysis shows that the training data does contain examples of very large fires. For example, the training set contains 48 fires above 10,000 ha, 26 above 15,000 ha, and 16 above 20,000 ha. Therefore, the testing set is not introducing an entirely unseen fire-size range.

The proportions of large fires are also very similar between training and testing. For example, fires above 5,000 ha represent approximately 1.32% of the training data and 1.11% of the testing data. Similarly, fires above 20,000 ha represent approximately 0.18% of training observations and 0.13% of testing observations.

These results indicate that the relatively low predictive performance cannot be explained simply by an inappropriate train-test split or the presence of completely unseen extreme fire sizes. The dataset contains representative examples of large fires in both subsets.

The difference in maximum values is therefore considered a consequence of random sampling rather than evidence of severe distribution mismatch. The investigation should now focus on whether the available environmental predictors contain sufficient information to explain variations in burned-area severity.

In [ ]:
# EXTREME-FIRE PREDICTABILITY AUDIT

# Predictions from our current champion model
champion_test_pred = best_xgb.predict(X_test)

# Convert predictions and actual values back to hectares
champion_pred_ha = np.expm1(champion_test_pred)
actual_test_ha = np.expm1(y_test)

# Prevent negative predictions
champion_pred_ha = np.maximum(champion_pred_ha, 0)

# Create evaluation dataframe
extreme_analysis = pd.DataFrame({
    "Actual_ha": actual_test_ha,
    "Predicted_ha": champion_pred_ha
})

# Define fire-size groups
def fire_range(area):
    if area <= 100:
        return "30–100 ha"
    elif area <= 500:
        return "101–500 ha"
    elif area <= 5000:
        return "501–5,000 ha"
    elif area <= 10000:
        return "5,001–10,000 ha"
    else:
        return ">10,000 ha"

extreme_analysis["Fire_Range"] = (
    extreme_analysis["Actual_ha"]
    .apply(fire_range)
)

# Calculate group statistics
range_results = (
    extreme_analysis
    .groupby("Fire_Range", sort=False)
    .apply(
        lambda g: pd.Series({
            "Records": len(g),
            "Mean_Actual_ha": g["Actual_ha"].mean(),
            "Mean_Predicted_ha": g["Predicted_ha"].mean(),
            "MAE_ha": mean_absolute_error(
                g["Actual_ha"],
                g["Predicted_ha"]
            ),
            "Mean_Error_ha": (
                g["Actual_ha"] - g["Predicted_ha"]
            ).mean()
        }),
        include_groups=False
    )
    .reset_index()
)

display(
    range_results.round(2)
)

,Fire_Range,Records,Mean_Actual_ha,Mean_Predicted_ha,MAE_ha,Mean_Error_ha
0,30–100 ha,918.0,58.14,144.35,86.62,-86.21
1,101–500 ha,971.0,222.15,200.21,108.19,21.94
2,"501–5,000 ha",347.0,1301.07,293.62,1013.32,1007.45
3,">10,000 ha",10.0,16222.80,610.49,15612.31,15612.31
4,"5,001–10,000 ha",15.0,7128.47,494.17,6634.30,6634.30


### Interpretation: Extreme-Fire Predictability Audit

The extreme-fire predictability analysis reveals a strong systematic underprediction of large and extreme burned areas.

For relatively small fires between 30 and 100 ha, the model predicts an average of 144.35 ha compared with an actual average of 58.14 ha. Although this represents some overprediction, the absolute error remains relatively limited compared with the larger fire categories.

For fires between 101 and 500 ha, the model performs considerably better, predicting an average of 200.21 ha compared with an actual average of 222.15 ha.

However, prediction performance deteriorates substantially for larger fires. For the 501–5,000 ha category, the actual mean burned area is 1,301.07 ha while the model predicts only 293.62 ha, resulting in a mean underprediction of approximately 1,007 ha.

The problem becomes much more severe for fires above 5,000 ha. For the 5,001–10,000 ha category, the actual mean is 7,128.47 ha while the predicted mean is only 494.17 ha. For fires greater than 10,000 ha, the actual mean is 16,222.80 ha while the predicted mean is only 610.49 ha.

These results demonstrate that the model is strongly biased toward moderate burned-area values and fails to adequately represent extreme fire severity. The problem is therefore not simply a general lack of predictive ability; it is specifically a failure to distinguish and accurately model large and extreme fire events.

This finding explains why the overall Test R² remains relatively low despite extensive model optimisation. The small number of extreme observations has a very large influence on the regression error, while the model systematically pulls their predictions toward the much more common moderate-fire range.

The results suggest that further conventional hyperparameter tuning alone is unlikely to provide a substantial improvement. A specialised modelling strategy that first identifies potentially extreme fires and then applies a dedicated regression model to estimate their severity should therefore be investigated.


In [ ]:
# LARGE-FIRE CLASSIFICATION TARGET

LARGE_FIRE_THRESHOLD = 5000

y_train_large = (
    y_train_raw > LARGE_FIRE_THRESHOLD
).astype(int)

y_test_large = (
    y_test_raw > LARGE_FIRE_THRESHOLD
).astype(int)

print("=" * 60)
print("LARGE-FIRE CLASSIFICATION TARGET")
print("=" * 60)

print(f"Threshold: > {LARGE_FIRE_THRESHOLD:,} ha")

print("\nTraining:")
print(f"Normal fires : {(y_train_large == 0).sum()}")
print(f"Large fires  : {(y_train_large == 1).sum()}")

print("\nTesting:")
print(f"Normal fires : {(y_test_large == 0).sum()}")
print(f"Large fires  : {(y_test_large == 1).sum()}")

print("\nLarge-fire percentage:")
print(
    f"Training: {y_train_large.mean() * 100:.3f}%"
)

print(
    f"Testing : {y_test_large.mean() * 100:.3f}%"
)

LARGE-FIRE CLASSIFICATION TARGET
Threshold: > 5,000 ha

Training:
Normal fires : 8923
Large fires  : 119

Testing:
Normal fires : 2236
Large fires  : 25

Large-fire percentage:
Training: 1.316%
Testing : 1.106%


### Interpretation: Large-Fire Classification Target

A large-fire classification target was created using a threshold of greater than 5,000 hectares. The training dataset contains 119 large-fire observations and 8,923 normal-fire observations, while the testing dataset contains 25 large-fire observations and 2,236 normal-fire observations.

Large fires therefore represent approximately 1.32% of the training data and 1.11% of the testing data. The similar proportions between the two datasets indicate that the class distribution is reasonably consistent between training and testing.

However, the large-fire class is highly imbalanced, with approximately 99% of observations belonging to the normal-fire class. Therefore, a standard classifier could achieve a deceptively high accuracy simply by predicting almost every observation as a normal fire.

For this reason, overall accuracy will not be used as the primary evaluation metric. Precision, recall, F1-score, confusion matrix, and particularly recall for the large-fire class will be examined.

The training dataset contains 119 large-fire examples, providing a sufficient basis for an experimental severity classifier, although the relatively small number of extreme observations remains an important limitation.

In [43]:
# LARGE-FIRE XGBOOST CLASSIFIER

# Calculate class imbalance weight
negative_count = (y_train_large == 0).sum()
positive_count = (y_train_large == 1).sum()

scale_pos_weight = negative_count / positive_count

print("=" * 60)
print("LARGE-FIRE XGBOOST CLASSIFIER")
print("=" * 60)

print(f"Negative samples : {negative_count}")
print(f"Positive samples : {positive_count}")
print(f"Scale pos weight : {scale_pos_weight:.2f}")


large_fire_classifier = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=4,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1
)

print("\nTraining large-fire classifier...")

large_fire_classifier.fit(
    X_train,
    y_train_large,
    eval_set=[(X_test, y_test_large)],
    verbose=False
)

print("Large-fire classifier trained successfully.")

LARGE-FIRE XGBOOST CLASSIFIER
Negative samples : 8923
Positive samples : 119
Scale pos weight : 74.98

Training large-fire classifier...
Large-fire classifier trained successfully.


In [44]:
# EVALUATE LARGE-FIRE CLASSIFIER

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

# Predict probabilities
large_fire_prob = large_fire_classifier.predict_proba(
    X_test
)[:, 1]

# Default classification threshold
large_fire_pred = (
    large_fire_prob >= 0.50
).astype(int)

print("=" * 60)
print("LARGE-FIRE CLASSIFIER PERFORMANCE")
print("=" * 60)

print("\nClassification Report:")
print(
    classification_report(
        y_test_large,
        large_fire_pred,
        target_names=[
            "Normal Fire",
            "Large Fire"
        ],
        digits=4,
        zero_division=0
    )
)

print("\nConfusion Matrix:")

cm = confusion_matrix(
    y_test_large,
    large_fire_pred
)

cm_df = pd.DataFrame(
    cm,
    index=["Actual Normal", "Actual Large"],
    columns=["Predicted Normal", "Predicted Large"]
)

display(cm_df)

print("\nROC-AUC:")
print(
    round(
        roc_auc_score(
            y_test_large,
            large_fire_prob
        ),
        4
    )
)

print("\nPR-AUC:")
print(
    round(
        average_precision_score(
            y_test_large,
            large_fire_prob
        ),
        4
    )
)

LARGE-FIRE CLASSIFIER PERFORMANCE

Classification Report:
              precision    recall  f1-score   support

 Normal Fire     0.9927    0.9776    0.9851      2236
  Large Fire     0.1525    0.3600    0.2143        25

    accuracy                         0.9708      2261
   macro avg     0.5726    0.6688    0.5997      2261
weighted avg     0.9834    0.9708    0.9766      2261


Confusion Matrix:


,Predicted Normal,Predicted Large
Actual Normal,2186,50
Actual Large,16,9



ROC-AUC:
0.7821

PR-AUC:
0.1406


### Interpretation: Large-Fire Classifier

The large-fire XGBoost classifier achieved an overall accuracy of 97.08%. However, overall accuracy is not an appropriate primary metric for this experiment because the dataset is highly imbalanced, with large fires representing only 1.11% of the testing observations.

For the large-fire class, the classifier achieved a precision of 15.25%, recall of 36.00%, and F1-score of 21.43%. The recall indicates that the classifier successfully identified 36% of the large fires in the test set. However, the relatively low precision indicates that many observations classified as large fires were actually normal fires.

The classifier therefore demonstrates some ability to distinguish extreme fire conditions, but its performance is currently insufficient for reliable large-fire detection. Nevertheless, the result is important because it confirms that the environmental features contain some information related to extreme-fire occurrence.

The classifier should therefore not yet be integrated into the final prediction pipeline. Further threshold optimisation and evaluation are required before determining whether the severity-aware modelling approach can improve the burned-area regression model.

### Optimize the Large-Fire Detection Threshold

In [45]:
# SECTION 9.6 — LARGE-FIRE THRESHOLD ANALYSIS

from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [
    0.50,
    0.40,
    0.30,
    0.25,
    0.20,
    0.15,
    0.10,
    0.05
]

threshold_results = []

for threshold in thresholds:

    predictions = (
        large_fire_prob >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": threshold,

        "Large Fire Precision":
            precision_score(
                y_test_large,
                predictions,
                zero_division=0
            ),

        "Large Fire Recall":
            recall_score(
                y_test_large,
                predictions,
                zero_division=0
            ),

        "Large Fire F1":
            f1_score(
                y_test_large,
                predictions,
                zero_division=0
            ),

        "Predicted Large Fires":
            predictions.sum()
    })

threshold_results = pd.DataFrame(
    threshold_results
)

display(
    threshold_results.round(4)
)

,Threshold,Large Fire Precision,Large Fire Recall,Large Fire F1,Predicted Large Fires
0,0.50,0.1525,0.36,0.2143,59
1,0.40,0.1000,0.36,0.1565,90
2,0.30,0.0658,0.40,0.1130,152
3,0.25,0.0649,0.48,0.1143,185
4,0.20,0.0500,0.48,0.0906,240
5,0.15,0.0441,0.52,0.0812,295
6,0.10,0.0332,0.52,0.0624,392
7,0.05,0.0264,0.68,0.0508,644


### Interpretation: Large-Fire Threshold Analysis

Different probability thresholds were evaluated to determine whether the large-fire classifier could improve detection of extreme fire events.

At the default threshold of 0.50, the classifier achieved a large-fire recall of 36%, identifying 9 of the 25 large fires. Lowering the threshold increased recall, but this came at the cost of substantially reduced precision.

At a threshold of 0.25, recall increased to 48%, meaning that approximately 12 of the 25 large fires were detected. However, precision decreased to 6.49%, meaning that only a small proportion of observations classified as large fires were actually large fires.

The lowest threshold of 0.05 achieved the highest recall of 68%, identifying approximately 17 of the 25 large fires. However, precision decreased to only 2.64%, with 644 observations classified as large fires. This would generate an excessive number of false alarms and would therefore be unsuitable for practical deployment.

Overall, lowering the classification threshold can improve the detection of extreme fires, but the resulting false-positive rate becomes too high. The threshold analysis therefore does not provide a sufficiently reliable large-fire classifier for direct use in the final prediction system.

The results demonstrate that the available environmental features provide some information about extreme-fire occurrence, but they are not sufficiently discriminative to reliably separate large fires from normal fires. Therefore, the two-stage classification approach should not replace the existing regression model at this stage.

In [47]:
# FEATURE AVAILABILITY AUDIT

print("=" * 70)
print("CURRENT MODEL FEATURE SET")
print("=" * 70)

for i, feature in enumerate(X_train.columns, start=1):
    print(f"{i:2d}. {feature}")

print("\nTotal features:", X_train.shape[1])

CURRENT MODEL FEATURE SET
 1. latitude
 2. longitude
 3. elevation
 4. slope_degrees
 5. aspect
 6. curvature
 7. roads_distance_km
 8. population
 9. temperature_c
10. dew_point_c
11. relative_humidity
12. wind_speed
13. rainfall_mm
14. surface_pressure
15. solar_radiation
16. soil_moisture
17. ndvi
18. lai
19. lc_agriculture
20. lc_forest
21. lc_grassland
22. lc_settlement
23. lc_shrubland
24. lc_sparse_vegetation
25. lc_water_bodies
26. lc_wetland
27. year
28. month
29. day_of_year
30. month_sin
31. month_cos
32. day_of_year_sin
33. day_of_year_cos
34. temperature_c_missing
35. dew_point_c_missing
36. relative_humidity_missing
37. wind_speed_missing
38. rainfall_mm_missing
39. surface_pressure_missing
40. solar_radiation_missing
41. soil_moisture_missing
42. wind_direction_missing
43. wind_direction_sin
44. wind_direction_cos

Total features: 44


In [48]:
# NORMAL-FIRE REGRESSION DIAGNOSTIC

NORMAL_FIRE_THRESHOLD = 5000

print("=" * 70)
print("NORMAL-FIRE REGRESSION DIAGNOSTIC")
print("=" * 70)

print(f"Fire-size threshold: {NORMAL_FIRE_THRESHOLD:,} ha")

# Identify normal fires in training and testing data
train_normal_mask = y_train_raw <= NORMAL_FIRE_THRESHOLD
test_normal_mask = y_test_raw <= NORMAL_FIRE_THRESHOLD

# Create filtered datasets
X_train_normal = X_train.loc[train_normal_mask].copy()
X_test_normal = X_test.loc[test_normal_mask].copy()

y_train_normal = y_train.loc[train_normal_mask].copy()
y_test_normal = y_test.loc[test_normal_mask].copy()

y_train_normal_raw = y_train_raw.loc[train_normal_mask].copy()
y_test_normal_raw = y_test_raw.loc[test_normal_mask].copy()

print("\nNormal-fire observations:")
print(f"Training: {len(X_train_normal):,}")
print(f"Testing : {len(X_test_normal):,}")

print("\nExtreme-fire observations excluded:")
print(
    f"Training: {(~train_normal_mask).sum():,}"
)
print(
    f"Testing : {(~test_normal_mask).sum():,}"
)

NORMAL-FIRE REGRESSION DIAGNOSTIC
Fire-size threshold: 5,000 ha

Normal-fire observations:
Training: 8,923
Testing : 2,236

Extreme-fire observations excluded:
Training: 119
Testing : 25


In [50]:
# TRAIN NORMAL-FIRE XGBOOST

print("=" * 70)
print("TRAINING NORMAL-FIRE XGBOOST")
print("=" * 70)

normal_fire_xgb = xgb.XGBRegressor(
    **study.best_params,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

normal_fire_xgb.fit(
    X_train_normal,
    y_train_normal,
    eval_set=[
        (X_test_normal, y_test_normal)
    ],
    verbose=False
)

print("Normal-fire XGBoost trained successfully.")

TRAINING NORMAL-FIRE XGBOOST
Normal-fire XGBoost trained successfully.


In [51]:
# EVALUATE NORMAL-FIRE XGBOOST

normal_train_pred = normal_fire_xgb.predict(
    X_train_normal
)

normal_test_pred = normal_fire_xgb.predict(
    X_test_normal
)

normal_results = {
    "Model": "Normal-Fire XGBoost",

    "Train R²": r2_score(
        y_train_normal,
        normal_train_pred
    ),

    "Test R²": r2_score(
        y_test_normal,
        normal_test_pred
    ),

    "Train MAE": mean_absolute_error(
        y_train_normal,
        normal_train_pred
    ),

    "Test MAE": mean_absolute_error(
        y_test_normal,
        normal_test_pred
    ),

    "Train RMSE": np.sqrt(
        mean_squared_error(
            y_train_normal,
            normal_train_pred
        )
    ),

    "Test RMSE": np.sqrt(
        mean_squared_error(
            y_test_normal,
            normal_test_pred
        )
    )
}

display(
    pd.DataFrame([normal_results]).round(4)
)

,Model,Train R²,Test R²,Train MAE,Test MAE,Train RMSE,Test RMSE
0,Normal-Fire XGBoost,0.5777,0.2166,0.5813,0.7965,0.7217,0.9835


### Interpretation: Normal-Fire Regression Diagnostic

The Normal-Fire XGBoost model was trained after temporarily excluding fires larger than 5,000 hectares from both the training and testing datasets. The purpose of this experiment was to determine whether extreme-fire observations were solely responsible for the relatively low predictive performance of the original model.

The model achieved a Training R² of 0.5777 and a Testing R² of 0.2166. The Test MAE was 0.7965 and the Test RMSE was 0.9835.

Although extreme fires were removed, the Testing R² decreased from 0.2451 for the full Optimized XGBoost model to 0.2166 for the Normal-Fire model. This demonstrates that simply removing extreme-fire observations does not improve the general predictive relationship between the environmental predictors and burned area.

The experiment therefore provides evidence that extreme fires are not the only source of prediction difficulty. Even when the extreme tail is removed, the available environmental features do not explain enough of the variation in burned area to produce a substantially stronger regression model.

The results support the conclusion that further improvement is more likely to come from additional informative predictors or improved data quality rather than simply removing extreme fire observations.

In [52]:
# TEMPORAL DATA AVAILABILITY

print("=" * 70)
print("TEMPORAL DATA AVAILABILITY AUDIT")
print("=" * 70)

# Combine training and testing observations
all_features = pd.concat(
    [X_train, X_test],
    axis=0
)

print(f"\nTotal observations: {len(all_features):,}")

# Count unique geographic locations
unique_locations = (
    all_features[
        ["latitude", "longitude"]
    ]
    .drop_duplicates()
)

print(f"Unique locations: {len(unique_locations):,}")

print(
    f"Average observations per location: "
    f"{len(all_features) / len(unique_locations):.2f}"
)

# Count locations appearing more than once
location_counts = (
    all_features
    .groupby(
        ["latitude", "longitude"]
    )
    .size()
    .reset_index(name="observation_count")
)

repeated_locations = location_counts[
    location_counts["observation_count"] > 1
]

print(
    f"\nLocations appearing more than once: "
    f"{len(repeated_locations):,}"
)

print(
    f"Percentage of locations repeated: "
    f"{100 * len(repeated_locations) / len(unique_locations):.2f}%"
)

print("\nObservation-count distribution:")

display(
    location_counts["observation_count"]
    .describe()
    .round(2)
    .to_frame("Observations per Location")
)

TEMPORAL DATA AVAILABILITY AUDIT

Total observations: 11,303
Unique locations: 10,980
Average observations per location: 1.03

Locations appearing more than once: 311
Percentage of locations repeated: 2.83%

Observation-count distribution:


,Observations per Location
count,10980.00
mean,1.03
std,0.18
min,1.00
25%,1.00
50%,1.00
75%,1.00
max,3.00


In [53]:
# SECTION 11.2 — CHECK TEMPORAL VARIATION

# Add an observation identifier
temporal_check = all_features[
    [
        "latitude",
        "longitude",
        "year",
        "month",
        "day_of_year"
    ]
].copy()

# Number of unique dates per location
date_variation = (
    temporal_check
    .groupby(
        ["latitude", "longitude"]
    )
    .agg(
        unique_years=("year", "nunique"),
        unique_months=("month", "nunique"),
        unique_days=("day_of_year", "nunique"),
        observations=("day_of_year", "size")
    )
    .reset_index()
)

print("=" * 70)
print("TEMPORAL VARIATION BY LOCATION")
print("=" * 70)

print("\nLocations with multiple years:")
print(
    (date_variation["unique_years"] > 1).sum()
)

print("\nLocations with multiple months:")
print(
    (date_variation["unique_months"] > 1).sum()
)

print("\nLocations with multiple observation days:")
print(
    (date_variation["unique_days"] > 1).sum()
)

print("\nSummary:")
display(
    date_variation[
        [
            "unique_years",
            "unique_months",
            "unique_days",
            "observations"
        ]
    ]
    .describe()
    .round(2)
)

TEMPORAL VARIATION BY LOCATION

Locations with multiple years:
291

Locations with multiple months:
207

Locations with multiple observation days:
307

Summary:


,unique_years,unique_months,unique_days,observations
count,10980.00,10980.00,10980.00,10980.00
mean,1.03,1.02,1.03,1.03
std,0.17,0.14,0.17,0.18
min,1.00,1.00,1.00,1.00
25%,1.00,1.00,1.00,1.00
50%,1.00,1.00,1.00,1.00
75%,1.00,1.00,1.00,1.00
max,3.00,3.00,3.00,3.00


### Ierpretation: Temporal Data Availability

The temporal availability audit shows that the current dataset contains 11,303 observations representing 10,980 unique geographic locations. The average number of observations per location is only 1.03, indicating that most locations occur only once in the dataset.

Only 311 locations (2.83%) appear more than once, and no location contains more than three observations. Furthermore, only 291 locations have observations spanning multiple years, while 207 locations span multiple months and 307 locations contain observations from multiple days.

These results demonstrate that the dataset does not contain sufficient repeated temporal observations at the same geographic locations to reliably construct historical weather features such as 7-day rainfall accumulation, 14-day rainfall, 30-day rainfall, consecutive dry days, rolling temperature, or rolling humidity.

Therefore, temporal feature engineering based solely on the existing observations would introduce unreliable or misleading information. The current dataset should not be artificially expanded by assuming that observations from nearby locations represent the historical conditions of the same location.

The temporal audit therefore identifies an important limitation of the current dataset: it provides extensive spatial and environmental information, but very limited temporal continuity. This limitation may contribute to the difficulty of predicting burned-area severity, particularly for large fire events.

The results support augmenting the existing dataset with historical weather and drought-related information rather than continuing to perform extensive hyperparameter tuning on the same feature set.